# Data Cleaning
## Fix data quality issues found in data audit

**Goal:** Use problems from audit to:
- Fix data types
- Handle missing values
- Remove invalid data
- Create useful columns

**Output:** Clean datasets ready for analysis

<hr>

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

<hr>

## Load All Datasets

In [2]:
customers = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_customers_dataset.csv")
geolocation = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_geolocation_dataset.csv")
order_items = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_order_items_dataset.csv")
payments = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_order_payments_dataset.csv")
reviews = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_order_reviews_dataset.csv")
orders = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_orders_dataset.csv")
products = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_products_dataset.csv")
sellers = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\olist_sellers_dataset.csv")
category_translation = pd.read_csv(r"D:\Ayush\Learning\Projects\Ecommerce\data\raw\product_category_name_translation.csv")

print("***All datasets loaded successfully!***")

***All datasets loaded successfully!***


<hr>

## Convert Date Columns to Datetime

### Problem Found:
All date columns in orders and reviews are stored as strings (object type). This makes it hard to calculate time differences and work with dates.

### Solution:
Convert all date columns from string → datetime format

In [3]:
print("*"*10, "CONVERTING DATES", "*"*10)
print()

# Orders table
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

print("***Orders dates converted***\n")

# Reviews table
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

print("***Reviews dates converted***")
print()
print("-"*60)
print("\nNew data types:")
print(orders[['order_purchase_timestamp', 'order_delivered_customer_date']].dtypes)

********** CONVERTING DATES **********

***Orders dates converted***

***Reviews dates converted***

------------------------------------------------------------

New data types:
order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
dtype: object


### What This Does:
Now we can:
- Compare dates (which order arrived first?)
- Calculate days (how long to deliver?)
- Group by time periods (monthly revenue?)
- Find late deliveries (estimated vs actual)

<hr>

## Handle Missing Values

### Problem Found:
Several columns have missing values:
- Reviews: 88% missing titles, 59% missing messages
- Orders: ~3% missing delivery dates
- Products: 1.8% missing category info

### Solution:
Different approach for each problem

### Reviews: Missing Titles & Messages

In [4]:
print("*"*10, "REVIEWS MISSING VALUES", "*"*10)
print()
print(f"Missing review_comment_title: {reviews['review_comment_title'].isnull().sum()} ({reviews['review_comment_title'].isnull().sum()/len(reviews)*100:.1f}%)")
print(f"Missing review_comment_message: {reviews['review_comment_message'].isnull().sum()} ({reviews['review_comment_message'].isnull().sum()/len(reviews)*100:.1f}%)")
print()

# Decision: Many customers give rating but no text comment
# Fill missing values with 'No comment' to preserve the review
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No comment')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No comment')

print("***Filled missing review comments with 'No comment***")
print()
print(f"Missing after filling: {reviews['review_comment_title'].isnull().sum()}")

********** REVIEWS MISSING VALUES **********

Missing review_comment_title: 87656 (88.3%)
Missing review_comment_message: 58247 (58.7%)

***Filled missing review comments with 'No comment***

Missing after filling: 0


#### Why This Approach:
- 88% missing titles is NORMAL - not everyone writes a comment
- But we keep these reviews because rating is valid
- "No comment" acts as placeholder in analysis
- We don't lose data, just make it analyzable

### Orders: Missing Delivery Dates

In [5]:
print()
print("*"*10, "ORDERS MISSING DATES", "*"*10)
print()
print(f"Missing order_delivered_customer_date: {orders['order_delivered_customer_date'].isnull().sum()}")
print(f"Missing order_delivered_carrier_date: {orders['order_delivered_carrier_date'].isnull().sum()}")
print()

missing_delivery = orders[orders['order_delivered_customer_date'].isnull()]
print("Order status for missing delivery dates:")
print(missing_delivery['order_status'].value_counts())
print()

# Decision: These are orders that were CANCELLED or never delivered
# Keep them but mark separately (handled with delivery_status)
print("***Keeping missing delivery dates - they represent undelivered orders***")
print("   (Will be handled in delivery_status column)")


********** ORDERS MISSING DATES **********

Missing order_delivered_customer_date: 2965
Missing order_delivered_carrier_date: 1783

Order status for missing delivery dates:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

***Keeping missing delivery dates - they represent undelivered orders***
   (Will be handled in delivery_status column)


#### Why This Approach:
- Missing delivery date = order wasn't delivered
- Status column tells us WHY (cancelled, unavailable, etc.)
- Keep these orders because they matter for analysis
- Example: "What % of orders got cancelled?"

### Products: Missing Category

In [6]:
print()
print("*"*10, "PRODUCTS MISSING CATEGORY", "*"*10)
print()
print(f"Missing product_category_name: {products['product_category_name'].isnull().sum()}")
print()

# Decision: Products without category can't be analyzed
# Remove these rows since category is essential
products_before = len(products)
products = products[products['product_category_name'].notnull()]
products_after = len(products)

print(f"Removed {products_before - products_after} products without category")
print(f"Remaining products: {products_after}")
print()
print("***Products with missing category removed***")


********** PRODUCTS MISSING CATEGORY **********

Missing product_category_name: 610

Removed 610 products without category
Remaining products: 32341

***Products with missing category removed***


#### Why This Approach:
- Category is a KEY column for analysis
- "What's our top category?" requires category
- Can't fill it (we don't know what product it is)
- Better to remove 610 than keep incomplete data
- Still 32,341 products left

<hr>

## Remove Invalid Data

### Problem Found:
- Payments with value = 0 (9 records)
- Products with weight = 0 (possible)
- Review duplicates (814)

### Decision:
Remove records that don't make business sense

### Remove Zero Payment Values

In [7]:
print("*"*10, "INVALID PAYMENTS", "*"*10)
print()
print(f"Payments with value = 0: {(payments['payment_value'] == 0).sum()}")
print()

# Decision: Payment of 0 = cancelled or failed transaction
# Remove these because they distort revenue analysis
payments_before = len(payments)
payments = payments[payments['payment_value'] > 0]
payments_after = len(payments)

print(f"Removed {payments_before - payments_after} invalid payments")
print(f"Remaining payments: {payments_after}")
print()
print("***Zero-value payments removed***")

********** INVALID PAYMENTS **********

Payments with value = 0: 9

Removed 9 invalid payments
Remaining payments: 103877

***Zero-value payments removed***


#### Why This Approach:
- 0 payment = customer didn't actually pay
- Could be: cancelled order, failed transaction, refund
- These 9 records would give WRONG revenue totals
- Example: "Total revenue" would be inaccurate
- Better to remove 9 records than keep wrong data

### Check Product Zero Weights

In [8]:
print()
print("*"*10, "PRODUCT WEIGHTS", "*"*10)
print()
print(f"Products with weight = 0: {(products['product_weight_g'] == 0).sum()}")
print()

# Check if these are real products
zero_weight = products[products['product_weight_g'] == 0]
print("Categories of zero-weight products:")
print(zero_weight['product_category_name'].value_counts())
print()
print("***Zero-weight products kept - some are digital/service items (e.books, etc)***")


********** PRODUCT WEIGHTS **********

Products with weight = 0: 4

Categories of zero-weight products:
product_category_name
cama_mesa_banho    4
Name: count, dtype: int64

***Zero-weight products kept - some are digital/service items (e.books, etc)***


#### Decision:
- Digital products (e-books, services) have no physical weight
- These are VALID data, not errors
- Keep them and handle in analysis when needed

### Investigate Review Duplicates

In [9]:
print()
print("*"*10, "REVIEW DUPLICATES", "*"*10)
print()
print(f"Duplicate review_ids: {reviews['review_id'].duplicated().sum()}")
print()

dup_reviews = reviews[reviews['review_id'].duplicated(keep=False)]
print(f"Total rows with duplicate review_id: {len(dup_reviews)}")
print()
print("Sample duplicate review:")
print(dup_reviews[dup_reviews['review_id'] == dup_reviews['review_id'].iloc[0]][['review_id', 'order_id', 'review_score', 'review_creation_date']])


********** REVIEW DUPLICATES **********

Duplicate review_ids: 814

Total rows with duplicate review_id: 1603

Sample duplicate review:
                              review_id                          order_id  \
200    28642ce6250b94cc72bc85960aec6c62  e239d280236cdd3c40cb2c033f681d1c   
58385  28642ce6250b94cc72bc85960aec6c62  bc42a955f289870d5789e6e437206300   

       review_score review_creation_date  
200               5           2018-03-25  
58385             5           2018-03-25  


<br>

#### Findings:
- Same review_id appears multiple times
- But different order_ids - so different reviews
- This could be:
  - **Duplicate data entry (ERROR)** - remove it
  - **System issue** - same ID assigned twice

#### Decision for Now:
Keep duplicates as-is and flag for investigation. They may have different orders attached, so removing could lose valid data. Investigate further if analysis shows problems.

<hr>

## Create Useful New Columns

### What We'll Create:
- delivery_status - Simple status of delivery (Delivered, Not Delivered, etc)
- days_to_deliver - How many days from purchase to delivery
- delivery_delay - Days late (positive = late, negative = early)

### Create Delivery Status

In [10]:
print("*"*10, "CREATING DELIVERY STATUS", "*"*10)
print()

def get_delivery_status(status):
    if status == 'delivered':
        return 'Delivered'
    elif status in ['canceled', 'unavailable']:
        return 'Not Delivered'
    elif status in ['shipped', 'processing', 'invoiced', 'approved', 'created']:
        return 'In Progress'
    else:
        return 'Unknown'

orders['delivery_status'] = orders['order_status'].apply(get_delivery_status)

print("Delivery Status Distribution:")
print(orders['delivery_status'].value_counts())
print()
print("***delivery_status column created***")

********** CREATING DELIVERY STATUS **********

Delivery Status Distribution:
delivery_status
Delivered        96478
In Progress       1729
Not Delivered     1234
Name: count, dtype: int64

***delivery_status column created***


#### Why This Column:
- Simplifies analysis (only 3 categories instead of 8)
- Easy questions: "What % of orders delivered?"
- Can calculate: "Cancellation rate?"

### Calculate Days to Deliver

In [11]:
print()
print("*"*10, "CALCULATING DELIVERY TIME", "*"*10)
print()

delivered_mask = orders['delivery_status'] == 'Delivered'

orders['days_to_deliver'] = np.where(
    delivered_mask,
    (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days,
    np.nan
)

print(f"Orders with delivery time calculated: {orders['days_to_deliver'].notna().sum()}")
print()
print("Delivery Time Statistics:")
print(orders['days_to_deliver'].describe())
print()
print("***days_to_deliver column created***")


********** CALCULATING DELIVERY TIME **********

Orders with delivery time calculated: 96470

Delivery Time Statistics:
count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: days_to_deliver, dtype: float64

***days_to_deliver column created***


#### What This Tells Us:
- Average delivery: ~12 days
- Fastest: 1 day
- Slowest: ~200 days
- Helps identify bottlenecks or logistics issues

In [12]:
print()
print("*"*10, "CALCULATING DELIVERY DELAY", "*"*10)
print()

orders['delivery_delay_days'] = np.where(
    delivered_mask,
    (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days,
    np.nan
)

print("Delivery Delay Statistics:")
print(orders['delivery_delay_days'].describe())
print()

late_count = (orders['delivery_delay_days'] > 0).sum()
on_time_count = (orders['delivery_delay_days'] <= 0).sum()

print(f"Late deliveries (delay > 0 days): {late_count}")
print(f"On-time deliveries (delay <= 0): {on_time_count}")
print()
print("***delivery_delay_days column created***")


********** CALCULATING DELIVERY DELAY **********

Delivery Delay Statistics:
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

Late deliveries (delay > 0 days): 6534
On-time deliveries (delay <= 0): 89936

***delivery_delay_days column created***


#### Business Insight:
- Shows delivery performance
- Late deliveries might = lower customer ratings
- Questions: "Do late deliveries affect reviews?"

<hr>

## Summary of Changes

In [13]:
print("*"*10, "CLEANING SUMMARY", "*"*10)
print()
print("CHANGES MADE:")
print("-" * 60)
print()
print("CUSTOMERS - No changes needed")
print(f"   Rows: {len(customers)}")
print()
print("ORDERS - Added 3 new columns")
print(f"   Rows: {len(orders)}")
print(f"   New columns: delivery_status, days_to_deliver, delivery_delay_days")
print()
print("ORDER_ITEMS - No changes needed")
print(f"   Rows: {len(order_items)}")
print()
print("PAYMENTS - Change invalid payments")
print(f"   Rows: {len(payments)} (removed {payments_before - payments_after})")
print()
print("REVIEWS - Filled missing comments")
print(f"   Rows: {len(reviews)}")
print(f"   Missing comments filled: ~88% → 0%")
print()
print("PRODUCTS - Removed missing category")
print(f"   Rows: {len(products)} (removed {products_before - products_after})")
print()
print("SELLERS - No changes needed")
print(f"   Rows: {len(sellers)}")
print()
print("GEOLOCATION - No changes needed")
print(f"   Rows: {len(geolocation)}")
print()
print("CATEGORY_TRANSLATION - No changes needed")
print(f"   Rows: {len(category_translation)}")

********** CLEANING SUMMARY **********

CHANGES MADE:
------------------------------------------------------------

CUSTOMERS - No changes needed
   Rows: 99441

ORDERS - Added 3 new columns
   Rows: 99441
   New columns: delivery_status, days_to_deliver, delivery_delay_days

ORDER_ITEMS - No changes needed
   Rows: 112650

PAYMENTS - Change invalid payments
   Rows: 103877 (removed 9)

REVIEWS - Filled missing comments
   Rows: 99224
   Missing comments filled: ~88% → 0%

PRODUCTS - Removed missing category
   Rows: 32341 (removed 610)

SELLERS - No changes needed
   Rows: 3095

GEOLOCATION - No changes needed
   Rows: 1000163

CATEGORY_TRANSLATION - No changes needed
   Rows: 71


<hr>

## Save Cleaned Data

In [14]:
print("*"*10, "SAVING CLEANED DATA", "*"*10)
print()

path = (r"D:\Ayush\Learning\Projects\Ecommerce\data\cleaned\\")

customers.to_csv(path + "customers_cleaned.csv", index=False)
orders.to_csv(path + "orders_cleaned.csv", index=False)
order_items.to_csv(path + "order_items_cleaned.csv", index=False)
payments.to_csv(path + "payments_cleaned.csv", index=False)
reviews.to_csv(path + "reviews_cleaned.csv", index=False)
products.to_csv(path + "products_cleaned.csv", index=False)
sellers.to_csv(path + "sellers_cleaned.csv", index=False)
geolocation.to_csv(path + "geolocation_cleaned.csv", index=False)
category_translation.to_csv(path + "category_translation_cleaned.csv", index=False)

print("***All cleaned datasets saved to: ***")
print(f"   {path}")
print()
print("Files created:")
print("   - customers_cleaned.csv")
print("   - orders_cleaned.csv")
print("   - order_items_cleaned.csv")
print("   - payments_cleaned.csv")
print("   - reviews_cleaned.csv")
print("   - products_cleaned.csv")
print("   - sellers_cleaned.csv")
print("   - geolocation_cleaned.csv")
print("   - category_translation_cleaned.csv")
print()
print("***CLEANING COMPLETE! Ready for analysis.***")

********** SAVING CLEANED DATA **********

***All cleaned datasets saved to: ***
   D:\Ayush\Learning\Projects\Ecommerce\data\cleaned\\

Files created:
   - customers_cleaned.csv
   - orders_cleaned.csv
   - order_items_cleaned.csv
   - payments_cleaned.csv
   - reviews_cleaned.csv
   - products_cleaned.csv
   - sellers_cleaned.csv
   - geolocation_cleaned.csv
   - category_translation_cleaned.csv

***CLEANING COMPLETE! Ready for analysis.***
